# FastConformer fine-tune - Kaggle chạy code từ GitHub

Notebook này dùng cho Kaggle GPU. Nó clone repo main từ GitHub, thêm `src/` vào `PYTHONPATH`, cài runtime NeMo, rồi chạy trực tiếp module `asr_lab.train.finetune_vivos`. Không dùng code-dataset, không dùng wrapper local.


## Cần bật trong Kaggle

- Accelerator: GPU, ưu tiên T4 hoặc P100.
- Internet: On, vì notebook cần `git clone`, tải package, tải model và tải VIVOS.
- Commit notebook sau khi chạy xong để Kaggle lưu output trong `/kaggle/working`.


## Phương pháp bản main

- Repo chạy: `https://github.com/ManhTanTran/ASR_local.git`.
- Model nền: `nvidia/stt_en_fastconformer_transducer_large`.
- Recipe: đổi vocabulary tiếng Việt, fine-tune full encoder + decoder RNNT trên VIVOS.
- Output chính: `/kaggle/working/runs/vivos-fc115m-v2norm/results.json` và `.nemo`.


In [ ]:
from pathlib import Path
import json
import os
import shlex
import shutil
import subprocess
import sys

GITHUB_REPO = "https://github.com/ManhTanTran/ASR_local.git"
GITHUB_REF = "main"
REPO_DIR = Path("/kaggle/working/ASR_local")

RUN_ID = "vivos-fc115m-v2norm"
PRETRAINED = "nvidia/stt_en_fastconformer_transducer_large"
EPOCHS = 50
BATCH = 16
VOCAB_SIZE = 1024
LR = "2e-4"
PRECISION = "32"
MAX_MINUTES = 360
CONSOLE_LOG_STEPS = 25
CHECKPOINT_STEPS = 500
CHECKPOINT_KEEP = 2
AUTO_RESUME = True
RESUME_FROM_CHECKPOINT = ""  # optional path/glob .ckpt to force a specific resume file

RUN_DIR = Path("/kaggle/working/runs") / RUN_ID
LOG_PATH = RUN_DIR / "run.log"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"


def bootstrap_run(cmd, cwd=None, env=None, check=True):
    print("$", " ".join(shlex.quote(str(x)) for x in cmd), flush=True)
    return subprocess.run([str(x) for x in cmd], cwd=cwd, env=env, text=True, check=check)


print("GitHub repo:", GITHUB_REPO)
print("GitHub ref :", GITHUB_REF)
print("Run ID     :", RUN_ID)
print("Run log    :", LOG_PATH)
print("Checkpoints:", CHECKPOINT_DIR)


## Clone code main từ GitHub


In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

bootstrap_run(["git", "clone", "--depth", "1", "--branch", GITHUB_REF, GITHUB_REPO, REPO_DIR])
bootstrap_run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"])

SRC_DIR = REPO_DIR / "src"
sys.path.insert(0, str(SRC_DIR))

from asr_lab.common.run_logging import print_log_tail, run_logged

print("SRC_DIR =", SRC_DIR)
print("has asr_lab:", (SRC_DIR / "asr_lab").is_dir())
print("Run log =", LOG_PATH)
print("Checkpoint dir =", CHECKPOINT_DIR)


## Cài runtime cho Kaggle GPU


In [ ]:
# Cài torch cu118 để chạy được cả P100 sm_60 và T4 trên Kaggle.
# Không import/reload torch trong notebook kernel; train sẽ chạy trong subprocess.
run_logged([
    sys.executable, "-m", "pip", "-q", "install",
    "torch==2.7.1", "torchvision==0.22.1", "torchaudio==2.7.1",
    "--index-url", "https://download.pytorch.org/whl/cu118",
], log_path=LOG_PATH)
run_logged([sys.executable, "-m", "pip", "-q", "install", "nemo_toolkit[asr]==2.7.3"], log_path=LOG_PATH)


## Verify nhanh GPU + NeMo trong subprocess


In [ ]:
verify_code = """
import torch
import nemo
print('torch', torch.__version__)
print('cuda ', torch.cuda.is_available())
print('nemo ', nemo.__version__)
if torch.cuda.is_available():
    layer = torch.nn.LSTM(8, 8).cuda()
    x = torch.randn(2, 3, 8).cuda()
    _ = layer(x)
    print('LSTM-cuda-OK')
"""
run_logged([sys.executable, "-c", verify_code], log_path=LOG_PATH)


## Chạy fine-tune FastConformer theo main


In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = str(SRC_DIR) + os.pathsep + env.get("PYTHONPATH", "")
env["ASR_ARTIFACTS_DIR"] = "/kaggle/working"
env["ASR_DATA_DIR"] = "/tmp/vivos_data"
env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
env["PYTHONUNBUFFERED"] = "1"
env["ASR_CONSOLE_LOG_STEPS"] = str(CONSOLE_LOG_STEPS)
env["ASR_CHECKPOINT_STEPS"] = str(CHECKPOINT_STEPS)
env["ASR_CHECKPOINT_KEEP"] = str(CHECKPOINT_KEEP)
if RESUME_FROM_CHECKPOINT:
    env["ASR_RESUME_FROM_CHECKPOINT"] = RESUME_FROM_CHECKPOINT

cmd = [
    sys.executable,
    "-u",
    "-m",
    "asr_lab.train.finetune_vivos",
    "--pretrained",
    PRETRAINED,
    "--run-id",
    RUN_ID,
    "--epochs",
    str(EPOCHS),
    "--batch",
    str(BATCH),
    "--vocab-size",
    str(VOCAB_SIZE),
    "--lr",
    LR,
    "--precision",
    PRECISION,
    "--max-minutes",
    str(MAX_MINUTES),
    "--console-log-steps",
    str(CONSOLE_LOG_STEPS),
    "--checkpoint-steps",
    str(CHECKPOINT_STEPS),
    "--checkpoint-keep",
    str(CHECKPOINT_KEEP),
]
if RESUME_FROM_CHECKPOINT:
    cmd.extend(["--resume-from-checkpoint", RESUME_FROM_CHECKPOINT])
if not AUTO_RESUME:
    cmd.append("--no-auto-resume")

print("Run log:", LOG_PATH)
print("Checkpoint dir:", CHECKPOINT_DIR)
print("Auto resume:", AUTO_RESUME)
print("Resume override:", RESUME_FROM_CHECKPOINT or "none")
run_logged(cmd, cwd=REPO_DIR, env=env, log_path=LOG_PATH)


## Xem log run moi nhat

Sau khi clone repo, moi command chinh trong notebook dung `run_logged(...)` tu `asr_lab.common.run_logging` va ghi stdout/stderr vao `run.log`. Trong luc train, checkpoint dinh ky nam o `runs/<run_id>/checkpoints/*.ckpt`; file `checkpoint_manifest.json` cho biet checkpoint moi nhat. Neu output Kaggle khong tu cuon, chay cell nay de xem doan cuoi log.


In [ ]:
print_log_tail(LOG_PATH)


## Đọc kết quả WER ngay trên Kaggle


In [ ]:
run_dir = Path("/kaggle/working/runs") / RUN_ID
results_path = run_dir / "results.json"
status_path = run_dir / "status.json"
checkpoint_manifest_path = run_dir / "checkpoints" / "checkpoint_manifest.json"

print("run_dir:", run_dir)
print("status exists             :", status_path.exists())
print("results exists            :", results_path.exists())
print("checkpoint manifest exists:", checkpoint_manifest_path.exists())

if status_path.exists():
    print("\nSTATUS")
    print(json.dumps(json.loads(status_path.read_text()), ensure_ascii=False, indent=2))
if checkpoint_manifest_path.exists():
    print("\nCHECKPOINTS")
    print(json.dumps(json.loads(checkpoint_manifest_path.read_text()), ensure_ascii=False, indent=2))
if results_path.exists():
    print("\nRESULTS")
    results = json.loads(results_path.read_text())
    print(json.dumps(results, ensure_ascii=False, indent=2))


## Liệt kê artifact để tải về từ Kaggle Output


In [ ]:
for path in sorted(run_dir.rglob("*")):
    if path.is_file():
        size_mb = path.stat().st_size / 1_000_000
        print(f"{path.relative_to(run_dir)}\t{size_mb:.2f} MB")


## Sau khi chạy xong

Trong Kaggle, bấm `Save Version` hoặc `Commit` để lưu output. Artifact cần tải nằm trong phần Output của notebook, đặc biệt là thư mục `runs/vivos-fc115m-v2norm/` chứa `results.json`, `status.json`, `run.log`, `metrics.csv` và file `.nemo`.
